# Liveness 02 · Passive Model (multi-scale MiniFASNet + Fourier supervision)

**On-device passive anti-spoofing – most robust small design:**
- Two **MiniFASNetV2-SE** networks (depthwise CNN + squeeze-excitation, ≈0.45 M params each, 80×80 input),
  one on the face crop scaled **×2.7** (context: screen bezel, paper edge, hands) and one on **×4.0**.
- **Auxiliary Fourier-spectrum loss:** the net also predicts the log-magnitude FFT of its input – replay
  screens and prints have tell-tale frequency patterns (moiré, halftone), which boosts generalisation.
- Outputs averaged → live probability. Total < 2 MB, ≈10 ms.

Training mix: CelebA-Spoof + OULU + SiW + in-house; strong colour/blur/JPEG augmentation (different phones).

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Liveness
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"

class SE(nn.Module):
    def __init__(self, c, r=4):
        super().__init__(); self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // r, 1), nn.PReLU(), nn.Conv2d(c // r, c, 1), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)

def dw_block(i, o, s):
    return nn.Sequential(nn.Conv2d(i, i, 3, s, 1, groups=i, bias=False), nn.BatchNorm2d(i), nn.PReLU(i),
                         nn.Conv2d(i, o, 1, bias=False), nn.BatchNorm2d(o), nn.PReLU(o), SE(o))

class MiniFASNetV2SE(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.BatchNorm2d(32), nn.PReLU(32))
        self.body = nn.Sequential(dw_block(32, 32, 1), dw_block(32, 64, 2), dw_block(64, 64, 1),
                                  dw_block(64, 128, 2), dw_block(128, 128, 1), dw_block(128, 128, 1))
        self.ft_head = nn.Sequential(nn.Conv2d(128, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(), nn.Conv2d(64, 1, 3, 1, 1))  # Fourier branch
        self.cls = nn.Sequential(nn.Conv2d(128, 128, 5, groups=128, bias=False), nn.BatchNorm2d(128), nn.Flatten(),
                                 nn.Dropout(0.2), nn.Linear(128, num_classes))
    def forward(self, x):
        f = self.body(self.stem(x))               # 80x80 -> 10x10 -> cls conv 5x5 -> 6x6 pooled below
        logits = self.cls(F.adaptive_avg_pool2d(f, 5))
        return logits, self.ft_head(f)

m = MiniFASNetV2SE(); print(sum(p.numel() for p in m.parameters()) / 1e6, "M params")
print([t.shape for t in m(torch.zeros(2, 3, 80, 80))])

In [ ]:
def fourier_target(img_bgr, size=10):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    spec = np.log(np.abs(np.fft.fftshift(np.fft.fft2(gray))) + 1)
    spec = cv2.resize(spec, (size, size))
    return (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)

def scaled_crop(img, box, scale, out=80):
    x, y, w, h = box
    cx, cy, side = x + w / 2, y + h / 2, max(w, h) * scale
    x1, y1 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    crop = img[y1:int(cy + side / 2), x1:int(cx + side / 2)]
    return cv2.resize(crop, (out, out))

class Liveness_Crops(Dataset):
    """Manifest rows must carry face boxes (run YuNet from Face 02 once and store x,y,w,h columns)."""
    def __init__(self, manifest, split, scale):
        self.df, self.scale = manifest[manifest.split == split].reset_index(drop=True), scale
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(r.path)
        crop = scaled_crop(img, (r.x, r.y, r.w, r.h), self.scale)
        x = torch.from_numpy(crop[:, :, ::-1].copy()).permute(2, 0, 1).float() / 255
        return x, torch.tensor(int(r.label)), torch.from_numpy(fourier_target(crop)).unsqueeze(0).float()

In [ ]:
def train_scale(scale, epochs=25, lambda_ft=1.0):
    manifest = pd.read_csv(DATA / "Liveness" / "Manifest.csv")
    dl = DataLoader(Liveness_Crops(manifest, "train", scale), batch_size=256, shuffle=True, num_workers=4)
    model = MiniFASNetV2SE().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    for epoch in range(epochs):
        model.train()
        for x, y, ft in dl:
            x, y, ft = x.to(device), y.to(device), ft.to(device)
            logits, ft_pred = model(x)
            loss = F.cross_entropy(logits, y, label_smoothing=0.05) + lambda_ft * F.mse_loss(ft_pred, ft)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step(); print(f"scale {scale} epoch {epoch} loss {loss.item():.3f}")
    torch.save(model.state_dict(), RUNS / f"minifasnet_s{scale}.pt")
    return model

# m27 = train_scale(2.7); m40 = train_scale(4.0)